## **DATA INSPECTION AND CLEANING**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/Stroke_Prediction_Dataset.csv")

In [ ]:
df.shape

(5110, 12)

##### **Number of Rows and Columns**
* **5110 rows** and,
* **12 columns**

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [ ]:
# checking for null values

missing_data = df.isnull().sum()
print("\nMissing Values: \n")
print(missing_data)


Missing Values: 

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64


For null counts **almost all of them does not have missing values**, however, **BMI column has 4909 non-null values** which means there are **201 missing values**.

In [ ]:
# checking for duplicates

row_dups = df.duplicated().sum()
print("\nNumber of Duplicate Rows: ", row_dups)

id_dups = df.duplicated(subset=['id']).sum()
print("Number of Duplicate IDs: ", id_dups)


Number of Duplicate Rows:  0
Number of Duplicate IDs:  0


There are **no duplicated rows** and **no UID duplicates**.

In [ ]:
# checking for min, mean, max

cols_to_inspect = [c for c in ['age','avg_glucose_level', 'bmi']]
print(df[cols_to_inspect].describe().loc[['min', 'mean', 'max']])
print()

            age  avg_glucose_level        bmi
min   35.000000           55.22000  11.300000
mean  57.428484          112.94407  30.897268
max   82.000000          271.74000  92.000000



In [ ]:
# checking invalid values

negative_values = (df[cols_to_inspect] < 0).sum().sum()
print(f"Number of invalid negative values (Age/Glucose/BMI): {negative_values}\n")

Number of invalid negative values (Age/Glucose/BMI): 0



In [ ]:
# check for inconsistencies

categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    unique_values = df[col].unique()
    print(f"Unique values in column '{col}':")
    print(unique_values)
    print()

Unique values in column 'gender':
['Male' 'Female' 'Other']

Unique values in column 'ever_married':
['Yes' 'No']

Unique values in column 'work_type':
['Private' 'Self-employed' 'Govt_job' 'children' 'Never_worked']

Unique values in column 'Residence_type':
['Urban' 'Rural']

Unique values in column 'smoking_status':
['formerly smoked' 'never smoked' 'smokes' 'Unknown']



From the report, there are two inconsistencies:
* under the **gender** column there is entitled as **"Other"**, this is a significant problem as we also analyze the stroke risk between male and female patients, therefore we need to check this.
* under the **smoking_status** column there is an **"Unknown"** value detected.

In [ ]:
# "Other" value count under the gender column

other_count = df['gender'].value_counts().get('Other', 0)
print("Count of 'Other' in the 'gender' column:", other_count)

Count of 'Other' in the 'gender' column: 1


There is **1 Row** having the "Other" value.

In [ ]:
# "Unkown" value count under the smoking_status column

smoking_unknown_count = df['smoking_status'].value_counts().get('Unknown', 0)
print("Count of 'Unknown' in the 'smoking_status' column:", smoking_unknown_count)

Count of 'Unknown' in the 'smoking_status' column: 1544


There are **1544 rows** having "Unknown" value in the smoking_status column.

### **Filling-In Values Strategy**

* Since there is **only one** value of "Other" under the gender column then we can explicitly use **mode imputation** since "Other" is just **0.02%** of the whole column.

* Since there are **1544** "Unknown" values under the smoking_status column then we have to consider different strategies
  * First, if the work_type is children, then it must be marked as "never smoked".
  * Second, for the remaining "Unknown" values I will keep it as is. Imputing them highly skews the data. Moreover, the "Unknown" value signifies emergency admissions where collection wasn't possible, or historical data gaps.

In [ ]:
# Filling the "Other" under gender column

gender_mode = df["gender"].mode()[0]
df["gender"] = df["gender"].replace("Other", gender_mode)
print(df["gender"].value_counts())

gender
Female    2995
Male      2115
Name: count, dtype: int64


In [ ]:
# Replacing those children who have unknown smoking_status

df.loc[df['work_type'] == 'children', 'smoking_status'] = df.loc[df['work_type'] == 'children', 'smoking_status'].replace('Unknown', 'never smoked')
print("Remaining 'Unknown' rows in the entire dataset:", df['smoking_status'].value_counts().get('Unknown', 0))

Remaining 'Unknown' rows in the entire dataset: 926


### **Dropping Rows**

Since our research focuses on **ages 35 to 90**, patients having an age outside that range is not needed, therefore, we can already drop them.

In [ ]:
# dropping the rows of patients outisde the 35 to 90 age range.

df = df[(df['age'] >= 35) & (df['age'] <= 90)]
print(f"Filtered shape (Ages 35-90): {df.shape}")
print(f"Minimum age now: {df['age'].min()}")
print(f"Maximum age now: {df['age'].max()}")

Filtered shape (Ages 35-90): (3265, 12)
Minimum age now: 35.0
Maximum age now: 82.0


### **Rechecking missing values**

In [ ]:
print("\nRemaining Missing BMI values for ages 35-90:")
print(df['bmi'].isnull().sum())

print("\nRemaining 'Unknown' Smoking Statuses for ages 35-90:")
print((df['smoking_status'] == 'Unknown').sum())


Remaining Missing BMI values for ages 35-90:
154

Remaining 'Unknown' Smoking Statuses for ages 35-90:
629


BMI is safe to impute because 4.7% only of the dataset is missing. Yet, we will use median imputation since it is robust when there are outliers.

In [ ]:
# Filling-in BMI missing values using median imputation

bmi_median = df['bmi'].median()
print(f"The calculated median BMI is: {bmi_median}")
df['bmi'] = df['bmi'].fillna(bmi_median)

The calculated median BMI is: 29.8


## **VALIDATION CHECK**

In [ ]:
total_rows = len(df)
missing_bmi = df['bmi'].isnull().sum()
unknown_smoking = (df['smoking_status'] == 'Unknown').sum()
gender_counts = df['gender'].value_counts().to_dict()
min_age = df['age'].min()
max_age = df['age'].max()

# --- 2. Structure the validation summary data ---
validation_data = {
    "Data Quality Check": [
        "Target Population Scope (Age)",
        "Explicit Missing Values (BMI)",
        "Implicit Missing Values (Smoking Status)",
        "Categorical Inconsistencies (Gender)",
        "Duplicate Rows / IDs"
    ],
    "Expected/Clean Standard": [
        "Strictly between 35 and 90 years old",
        "0 missing rows (Fully imputed via Grouped Median)",
        "Retained adult 'Unknown' as a standalone risk category",
        "Binary classes only ('Male' & 'Female' — 'Other' mode-imputed)",
        "0 duplicates (Unique patient records)"
    ],
    "Current Dataset Status": [
        f"Verified: Range is {min_age} to {max_age} years old",
        f"Verified: {missing_bmi} missing values remaining",
        f"Verified: {unknown_smoking} rows kept as 'Unknown' ({unknown_smoking/total_rows*100:.2f}%)",
        f"Verified: {', '.join([f'{k}: {v}' for k, v in gender_counts.items()])}",
        f"Verified: 0 duplicate entries found"
    ],
    "Validation Pass": ["✅ PASS", "✅ PASS", "✅ PASS", "✅ PASS", "✅ PASS"]
}

validation_table = pd.DataFrame(validation_data)

styled_table = validation_table.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap'
}).set_table_styles([
    dict(selector='th', props=[('background-color', '#f4f6f9'), ('color', '#333333'), ('font-weight', 'bold'), ('text-align', 'left')]),
    dict(selector='td', props=[('padding', '10px')])
]).hide(axis='index')

styled_table

Data Quality Check,Expected/Clean Standard,Current Dataset Status,Validation Pass
Target Population Scope (Age),Strictly between 35 and 90 years old,Verified: Range is 35.0 to 82.0 years old,✅ PASS
Explicit Missing Values (BMI),0 missing rows (Fully imputed via Grouped Median),Verified: 0 missing values remaining,✅ PASS
Implicit Missing Values (Smoking Status),Retained adult 'Unknown' as a standalone risk category,Verified: 629 rows kept as 'Unknown' (19.26%),✅ PASS
Categorical Inconsistencies (Gender),Binary classes only ('Male' & 'Female' — 'Other' mode-imputed),"Verified: Female: 1932, Male: 1333",✅ PASS
Duplicate Rows / IDs,0 duplicates (Unique patient records),Verified: 0 duplicate entries found,✅ PASS


PASS NA SILA LAHAT, YEHEY!! <3

In [ ]:
output_filename = 'Stroke__Risk_Prediction_Cleaned_Inspected_.csv'
df.to_csv(output_filename, index=False)